In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
!git clone https://github.com/EauDeData/ODAOCR
!pip install easyocr
!cp -r ODAOCR/* .

Cloning into 'ODAOCR'...
remote: Enumerating objects: 141, done.
remote: Counting objects: 100% (141/141), done.
remote: Compressing objects: 100% (102/102), done.
remote: Total 141 (delta 54), reused 104 (delta 36), pack-reused 0 (from 0)
Receiving objects: 100% (141/141), 9.49 MiB | 11.77 MiB/s, done.
Resolving deltas: 100% (54/54), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 56.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 299.6/299.6 kB 28.7 MB/s eta 0:00:00


# Variables
Rutas que necesitaremos más adelante


In [ ]:
# Config variables
tokenizer_path = 'drive/MyDrive/TFM-Sara/assets/oda_giga_tokenizer'
image_folder = "drive/MyDrive/TFM-Sara/input/transcriptions"
results_folder = "drive/MyDrive/TFM-Sara/output/transcripciones"
handwritten_model_path = "drive/MyDrive/TFM-Sara/assets/handwritten_expert.pt"
custom_dictionary_path = 'drive/MyDrive/TFM-Sara/assets/transcription-examples.txt'

Cargamos el tokenizer


In [ ]:
from constructors import prepare_model, make_inference, GreedyTextDecoder, CharTokenizer
from PIL import Image

# Initialize tokenizer
# local_path: dummy_data/
# tokenizer_name: oda_giga_tokenizer
# The tokenizer itself does os.path.join(local_path/name + .json)
tokenizer = CharTokenizer(False, './', tokenizer_path)



/usr/local/lib/python3.12/dist-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)


Tokenizer drive/MyDrive/TFM-Sara/assets/oda_giga_tokenizer found in ./, loading tokens from local storage.


Cargamos el pt, si no existe arrastrar handwritten_expert_handwritten_expert (dentro de 111.zip) a la carpeta principal (/), cuidado que tarda un rato


In [ ]:
# Load model with tokenizer information
model = prepare_model(len(tokenizer), device='cuda', load_checkpoint=True,
                       checkpoint_name=handwritten_model_path)


Loaded model with: 152 modules.
Loading state dict from: drive/MyDrive/TFM-Sara/assets/handwritten_expert.pt
(I, script) Found incompatible keys: <All keys matched successfully>




```
# Tiene formato de código
```

Función que detecta textos de las fotos

In [ ]:
import easyocr
import numpy as np
from PIL import Image
import cv2
# Singleton: EasyOCR se inicializa una vez para toda la sesión
_easyocr_reader = None

def segment_words(image_path: str) -> list[Image.Image]:
    """
    Uses EasyOCR's CRAFT text detector to find word bounding boxes,
    returns a list of PIL image crops — one per detected word.
    Note: Catalan is not supported by EasyOCR; using Spanish ('es').
    """
    global _easyocr_reader
    if _easyocr_reader is None:
        _easyocr_reader = easyocr.Reader(['es'], gpu=True)
    reader = _easyocr_reader

    # detail=1 returns (bbox, text, confidence)
    # paragraph=False keeps individual word/token detections
    results = reader.readtext(image_path, detail=1, paragraph=False)

    pil_img = Image.open(image_path).convert("RGB")
    word_crops = []

    for (bbox, text, conf) in results:
        if not text.strip() or conf < 0.5:
            continue

        # bbox is [[x1,y1],[x2,y1],[x2,y2],[x1,y2]] (quad, not axis-aligned)
        xs = [p[0] for p in bbox]
        ys = [p[1] for p in bbox]
        x1, y1 = int(min(xs)), int(min(ys))
        x2, y2 = int(max(xs)), int(max(ys))

        # Ensure valid coordinates for cropping
        if x1 >= x2 or y1 >= y2:
            print(f"Warning: Invalid bounding box coordinates for text '{text}' in {image_path}. Skipping crop.")
            continue

        crop = pil_img.crop((x1, y1, x2, y2))

        # Check if the cropped image has valid dimensions
        if crop.width > 0 and crop.height > 0:
            word_crops.append(crop)
        else:
            print(f"Warning: Cropped image for text '{text}' has zero dimensions in {image_path}. Skipping.")

    return word_crops
def adaptive_threshold(image: Image.Image) -> Image.Image:
    gray = np.array(image.convert("L"))

    # Check if the gray image is empty or has zero dimensions
    if gray.size == 0 or gray.shape[0] == 0 or gray.shape[1] == 0:
        print(f"Warning: Input image to adaptive_threshold is empty or has zero dimensions. Returning original image.")
        return image # Return the original image or handle as appropriate

    blurred = cv2.GaussianBlur(gray, (17, 17), 0)
    binary = cv2.adaptiveThreshold(blurred, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY_INV, 21, 10)

    return Image.fromarray(binary)

Coge los textos recortados y intenta transcribir

---



---



In [ ]:
from constructors import GreedyTextDecoder, make_inference

def get_image_transcriptions(image_path: str, model, tokenizer, device: str) -> list[str]:
    """
    Transcribes all words detected in an image using the provided model and tokenizer.
    Encapsulates word segmentation, adaptive thresholding, and model inference.

    Args:
        image_path (str): Path to the input image file.
        model: The trained model for transcription.
        tokenizer: The tokenizer used for encoding/decoding text.
        device (str): The device to run inference on (e.g., 'cpu', 'cuda').

    Returns:
        list[str]: A list of transcribed words from the image.
    """
    word_crops = segment_words(image_path)

    transcribed_words = []

    decoder = GreedyTextDecoder()

    for word_img in word_crops:
        processed_word = adaptive_threshold(word_img)

        transcription = make_inference(model, tokenizer, decoder, processed_word, device)

        transcribed_words.append(transcription)

    return transcribed_words

Función principal donde recorre todas las fotos y aplica las funciones de arriba para transcribir, lo añade al array `results`

In [ ]:
import cv2
import os
import pandas as pd
from PIL import Image

results = []
device = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'

# ── Procesamiento incremental: saltar imágenes ya procesadas ──────────────
_already_done = set()
_csv_prev = os.path.join(results_folder, 'transcripciones.csv')
if os.path.exists(_csv_prev):
    try:
        import pandas as _pd_prev
        _df_prev = _pd_prev.read_csv(_csv_prev)
        if 'imagen' in _df_prev.columns:
            _already_done = set(_df_prev['imagen'].dropna().unique())
            print(f'Incremental: {len(_already_done)} imágenes ya procesadas → se saltarán')
    except Exception as _e:
        print(f'No se pudo leer CSV previo: {_e}')

# Ensure the image_folder exists
if not os.path.exists(image_folder):
    os.makedirs(image_folder)
    print(f"Created directory: {image_folder}")

# Verificar que hay imágenes en la carpeta antes de continuar
if not os.listdir(image_folder):
    raise FileNotFoundError(f"La carpeta de entrada '{image_folder}' está vacía. "
                            "Añade las fotos a analizar antes de ejecutar el pipeline.")


for filename in sorted(os.listdir(image_folder)):
    if filename.lower().endswith((".jpg", ".png", ".jpeg")):
        if filename in _already_done:
            continue

        image_path = os.path.join(image_folder, filename)

        img = cv2.imread(image_path)
        if img is None:
            results.append({
                "imagen": filename,
                "transcripcion": "ERROR: imagen no legible por OpenCV"
            })
            print(f"Error reading image: {image_path}")
            continue

        try:
            transcriptions_for_image = get_image_transcriptions(
                image_path,
                model,
                tokenizer,
                device=device
            )

            if transcriptions_for_image:
                for word_entry in transcriptions_for_image:
                    if word_entry and len(word_entry) > 0:
                        word = word_entry
                        results.append({
                            "imagen": filename,
                            "error":None,
                            "transcripcion_original": word,
                            "transcripcion_corrected": None,
                        })
                        print(f"Transcripcion para {filename}: {word}")
            else:
                results.append({
                    "imagen": filename,
                    "error": "No words transcribed",
                    "transcripcion_original": None,
                    "transcripcion_corrected": None,
                })
                print(f"No words transcribed for {filename}")


        except Exception as e:
            results.append({
                "imagen": filename,
                "error": str(e),
                "transcripcion_original": None,
                "transcripcion_corrected": None,
            })
            print(f"Error processing {filename}: {e}")

print("All images transcribed :)")

Incremental: 0 imágenes ya procesadas → se saltarán


Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% CompleteTranscripcion para IAAH_GUDIOL_34303.jpg: ['e8aba']
Transcripcion para IAAH_GUDIOL_34379.jpg: ['sartago']
Transcripcion para IAAH_GUDIOL_34383.jpg: ['t']
Transcripcion para IAAH_GUDIOL_34383.jpg: ['miacki']
Transcripcion para IAAH_GUDIOL_34383.jpg: ['des']
Transcripcion para IAAH_GUDIOL_34384.jpg: ['avlard']
Transcripcion para IAAH_GUDIOL_34384.jpg: ['manit']
Transcripcion para IAAH_GUDIOL_34384.jpg: ['vinwa']
Transcripcion para IAAH_GUDIOL_34389.jpg: ['panlaedoahini,ah']
No words transcribed for IAAH_GUDIOL_34390.jpg
Transcripcion para IAAH_GUDIOL_34392.jpg: ['t']
Transcripcion para IAAH_GUDIOL_34394.jpg: ['se&raba']
Transcripcion para IAAH_GUDIOL_34395.jpg: ['e']
Transcripcion para IAAH_GUDIOL_34395.jpg: ['baro']
Transcripcion para IAAH_GUDIOL_34396.jpg: ['a']
Transcripcion para IAAH_GUDIOL_34396.jpg: ['#']
Transcripcion para IAAH_GUDIOL_34396.jpg: ['avia']
Transcripcion para IAAH_GUDIOL_34396.jpg: ['penyar

In [ ]:
import re

# Define a set of common Spanish stop words (can be extended)
STOP_WORDS = set([
    # Castellano
    'de', 'en', 'el', 'la', 'los', 'las', 'un', 'una', 'unos', 'unas',
    'y', 'o', 'pero', 'a', 'ante', 'bajo', 'con', 'contra', 'desde',
    'durante', 'entre', 'hacia', 'hasta', 'mediante', 'para', 'por',
    'según', 'sin', 'sobre', 'tras', 'al', 'del', 'mi', 'su', 'es',
    'se', 'no', 'ni', 'si', 'me', 'te', 'le', 'nos', 'vos', 'os', 'les',
    'que', 'más', 'este', 'esta', 'esto', 'ese', 'esa', 'eso',
    # Català
    'i', 'amb', 'per', 'però', 'o', 'que', 'és', 'una', 'un', 'els', 'les',
    'del', 'dels', 'al', 'als', 'ha', 'han', 'hem', 'heu', 'havia', 'ser',
    'tot', 'tots', 'tota', 'totes', 'molt', 'més', 'ja', 'hi', 'ho', 'ne',
    # Símbolos y residuos
    ':', '.', ',', '-', '_', '|'])

def clean_final_word(word):
    """
    Cleans a single word by:
    - Handling None/NaN inputs.
    - Converting to lowercase.
    - Removing non-alphanumeric characters (keeping Spanish accents).
    - Removing single-character words.
    - Removing common stop words.
    """
    if pd.isna(word) or word is None:
        return None

    original_word_str = str(word)
    lower_word = original_word_str.lower()

    # Remove symbols, keeping alphanumeric characters and common Spanish accented letters
    cleaned_word = re.sub(r'[^a-záéíóúüñ0-9]', '', lower_word)

    # Remove if it's a single character or empty after cleaning
    if len(cleaned_word) <= 1:
        return None

    # Remove if it's a stop word
    if cleaned_word in STOP_WORDS:
        return None

    return cleaned_word


In [ ]:

pip install pyspellchecker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 38.4 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import os

df = pd.DataFrame(results)

if not os.path.exists(results_folder):
    os.makedirs(results_folder)
    print(f"Created directory: {results_folder}")



In [ ]:
import pandas as pd
from spellchecker import SpellChecker
import os

# Define the path to the custom dictionary
custom_dictionary_path = 'drive/MyDrive/TFM-Sara/assets/transcription-examples.txt'

# Initialize SpellChecker without a specific language to rely on the custom dictionary
spell = SpellChecker(language='es')

words_from_custom_dict_set = set() # Store custom words in a set for quick lookup

# Load the custom dictionary, using 'utf-8-sig' to handle Byte Order Mark (BOM)
if os.path.exists(custom_dictionary_path):
    with open(custom_dictionary_path, 'r', encoding='utf-8-sig') as f:
        loaded_words = [w.strip().lower() for w in f.read().splitlines() if w.strip()]
    words_from_custom_dict_set.update(loaded_words) # Add to the set

    # Load these words into the spell checker's frequency dictionary
    # This is the correct way to add them, and spell.correction() will use these frequencies.
    spell.word_frequency.load_words(loaded_words)

    print(f"Loaded custom dictionary from: {custom_dictionary_path} with {len(words_from_custom_dict_set)} unique words.") # DEBUG: Print loaded words
else:
    print(f"Custom dictionary not found at: {custom_dictionary_path}. Please create this file with correct words.")
    # If the file doesn't exist, the spell checker will operate with its default dictionary.

# Function to correct a single word
def correct_word(word):
    if pd.isna(word):
        return None

    original_word_str = str(word)
    lower_original_word = original_word_str.lower()

    # Strategy 1: If the lowercased original word is exactly in our custom dictionary, return it (preserving original case).
    if lower_original_word in words_from_custom_dict_set:
        # print(f"Original: {original_word_str}, Exact match in custom dict. Returning original.") # Verbose debug
        return original_word_str

    # Strategy 2: If not an exact match, try to correct using spell.correction()
    # This function already uses its frequency data (including our custom words) and internal distance calculations.
    correction = spell.correction(lower_original_word)

    # If a correction was found AND it's different from the original word
    # (and the correction is a known word to prevent weird suggestions)
    if correction and correction != lower_original_word and correction in spell.word_frequency:
        # spell.correction() will naturally prioritize custom words if they are plausible candidates
        # because their frequencies were loaded.
        # print(f"Original: {original_word_str}, Corrected to: {correction}. Returning it.") # Verbose debug
        return correction

    # Fallback: No strong correction found, return original word
    # print(f"Original: {original_word_str}, No strong correction found. Returning original.") # Verbose debug
    return original_word_str

# --- New Processing Order ---

# Verificar que el DataFrame tiene datos antes de continuar
if df.empty or 'transcripcion_original' not in df.columns:
    raise RuntimeError(
        f"Sin resultados de transcripción.\n"
        f"Causas posibles:\n"
        f"  · No hay imágenes .jpg/.png/.jpeg en '{image_folder}'\n"
        f"  · La celda de procesamiento no se ejecutó en esta sesión\n"
        f"  · Todas las imágenes lanzaron excepción durante el procesamiento\n"
        f"Columnas presentes: {list(df.columns)} — filas: {len(df)}"
    )

# 1. Apply initial cleaning to 'transcripcion_original'
df['transcripcion_pre_cleaned'] = df['transcripcion_original'].apply(clean_final_word)

# 2. Filter out rows where pre-cleaning resulted in None (i.e., 'garbage' words)
df_filtered = df[df['transcripcion_pre_cleaned'].notna()].copy()

# 3. Apply spell correction to the pre-cleaned words
# Cache de correcciones: evita llamar spell.correction() varias veces para la misma palabra
# (muy frecuente: "joaquim", "joan"... aparecen en decenas de fotos)
_correction_cache: dict = {}

def correct_word_cached(word):
    if word not in _correction_cache:
        _correction_cache[word] = correct_word(word)
    return _correction_cache[word]

df_filtered['transcripcion_corrected'] = df_filtered['transcripcion_pre_cleaned'].apply(correct_word_cached)
print(f"Palabras únicas corregidas: {len(_correction_cache)} (de {len(df_filtered)} filas)")

# 4. Apply final cleaning to the corrected words to get the 'transcripcion_final_cleaned'
df_filtered['transcripcion_final_cleaned'] = df_filtered['transcripcion_corrected'].apply(clean_final_word)

# Update the main DataFrame with the filtered and processed data
df = df_filtered.copy()


# Display a sample of the DataFrame to review the corrections
print("Sample of DataFrame with corrected and cleaned transcriptions:")
print(df.head())

# --- SAVE THE CORRECTED DATAFRAME ---
csv_path = os.path.join(results_folder, "transcripciones.csv")
json_path = os.path.join(results_folder, "transcripciones.json")

df.to_csv(csv_path, index=False, encoding="utf-8")
df.to_json(json_path, orient="records", indent=4, force_ascii=False)

print(f"Corrected and cleaned transcriptions saved to {csv_path} and {json_path}.")


Loaded custom dictionary from: drive/MyDrive/TFM-Sara/assets/transcription-examples.txt with 119 unique words.
Palabras únicas corregidas: 151 (de 157 filas)
Sample of DataFrame with corrected and cleaned transcriptions:
                  imagen error transcripcion_original transcripcion_corrected  \
0  IAAH_GUDIOL_34303.jpg  None                [e8aba]                   etapa   
1  IAAH_GUDIOL_34379.jpg  None              [sartago]                 saltado   
3  IAAH_GUDIOL_34383.jpg  None               [miacki]                   machi   
4  IAAH_GUDIOL_34383.jpg  None                  [des]                     des   
5  IAAH_GUDIOL_34384.jpg  None               [avlard]                   avaro   

  transcripcion_pre_cleaned transcripcion_final_cleaned  
0                     e8aba                       etapa  
1                   sartago                     saltado  
3                    miacki                       machi  
4                       des                         des  
5 

En vez de sobreescribir, juntamos lo viejo con los nuevos resultados (si hay fotos nuevas)

Juntamos los archivos

Ahora deberías tener ya un excel con los resultados de las transcripciones. Vamos a generar los grafos de conocimiento

In [ ]:
pip install networkx pyvis

In [ ]:
import pandas as pd
import os
import networkx as nx
from pyvis.network import Network

# Construct the full path to the 'transcripciones.csv' file
csv_file_path = os.path.join(results_folder, 'transcripciones.csv')

# Read the CSV file into a pandas DataFrame named df_transcriptions
df_transcriptions = pd.read_csv(csv_file_path)



# 1. Create an empty graph using NetworkX
G = nx.Graph()

# 2. Add nodes and edges from the df_transcriptions DataFrame
for index, row in df_transcriptions.iterrows():
    image_name = row['imagen'].strip()
    _corrected = row.get('transcripcion_corrected')
    _original  = row.get('transcripcion_original')
    _word = _corrected if pd.notna(_corrected) else _original
    if pd.isna(_word) or not str(_word).strip():
        continue
    transcribed_word = str(_word).strip()

    # Add image node (distinct images)
    if image_name not in G:
        G.add_node(image_name, label=image_name, title=image_name, node_type='foto', pk=image_name, group=1, size=25, color='#4A90D9', dimension='imagen')

    # Add word node (distinct transcribed words)
    if transcribed_word not in G:
        G.add_node(transcribed_word, label=transcribed_word, title=transcribed_word, node_type='palabra', pk=transcribed_word, group=2, size=15, color='#5CB85C', dimension='transcripcion')

    # Add an edge between the image and the transcribed word
    G.add_edge(image_name, transcribed_word, relation='contiene_palabra', dimension='transcripcion')

# 3. Initialize Pyvis Network
net = Network(notebook=True, height="750px", width="100%", cdn_resources='remote')

# Add nodes and edges from the NetworkX graph to the Pyvis network
# Nodes can have properties like 'title' (tooltip), 'group' (for different colors/shapes), 'size'
# Edges can have properties like 'title' (tooltip)

    for node in G.nodes:
    attrs = dict(G.nodes[node])  # copia para no modificar el grafo original
    attrs.pop('label', None)     # quita 'label' si existe
    net.add_node(node, label=node, **attrs)
    for node in G.nodes:
    attrs = G.nodes[node]
    net.add_node(node, **attrs)  # 'label' ya está en attrs


for u, v, data in G.edges(data=True):
    net.add_edge(u, v, title=data.get('relation', ''))

# Set physics options for better visualization
net.toggle_physics(True)

# Generate HTML and save to the results folder
output_filename = os.path.join(results_folder, "knowledge_graph.html")
net.show(output_filename)

print(f"Interactive knowledge graph saved to: {output_filename}")

IndentationError: unexpected indent (2510729899.py, line 45)

In [ ]:
import pandas as pd
import os
import networkx as nx
from pyvis.network import Network

# Construct the full path to the 'transcripciones.csv' file
csv_file_path = os.path.join(results_folder, 'transcripciones.csv')
df_transcriptions = pd.read_csv(csv_file_path)

# 1. Create an empty graph using NetworkX
G = nx.Graph()

# 2. Add nodes and edges from the df_transcriptions DataFrame
for index, row in df_transcriptions.iterrows():
    image_name = row['imagen'].strip()
    _corrected = row.get('transcripcion_corrected')
    _original  = row.get('transcripcion_original')
    _word = _corrected if pd.notna(_corrected) else _original
    if pd.isna(_word) or not str(_word).strip():
        continue
    transcribed_word = str(_word).strip()

    if image_name not in G:
        G.add_node(image_name, label=image_name, title=image_name, node_type='foto',
                   pk=image_name, group=1, size=25, color='#4A90D9', dimension='imagen')

    if transcribed_word not in G:
        G.add_node(transcribed_word, label=transcribed_word, title=transcribed_word,
                   node_type='palabra', pk=transcribed_word, group=2, size=15,
                   color='#5CB85C', dimension='transcripcion')

    G.add_edge(image_name, transcribed_word, relation='contiene_palabra', dimension='transcripcion')

# 3. Initialize Pyvis Network
net = Network(notebook=True, height="750px", width="100%", cdn_resources='remote')

# 4. Add nodes from NetworkX to Pyvis  ← solo UN bucle, sin duplicados
for node in G.nodes:
    attrs = dict(G.nodes[node])  # copia para no modificar el grafo original
    attrs.pop('label', None)     # evita pasar 'label' dos veces
    net.add_node(node, label=node, **attrs)

# 5. Add edges
for u, v, data in G.edges(data=True):
    net.add_edge(u, v, title=data.get('relation', ''))

# 6. Physics and output
net.toggle_physics(True)
output_filename = os.path.join(results_folder, "knowledge_graph.html")
net.show(output_filename)
print(f"Interactive knowledge graph saved to: {output_filename}")

drive/MyDrive/TFM-Sara/output/transcripciones/knowledge_graph.html
Interactive knowledge graph saved to: drive/MyDrive/TFM-Sara/output/transcripciones/knowledge_graph.html


In [ ]:
import os
import networkx as nx

# Escalar tamaño de nodos-palabra según su frecuencia (nº de imágenes que la contienen)
word_degrees = {n: deg for n, deg in G.degree()
                if G.nodes[n].get('dimension') == 'transcripcion'}
max_freq = max(word_degrees.values(), default=1)
for node, deg in word_degrees.items():
    # Rango visual: 10 (rara) → 40 (muy frecuente)
    G.nodes[node]['size'] = 10 + 30 * (deg / max_freq)

# Construct the full path for the GEXF file
gexf_file_path = os.path.join(results_folder, 'knowledge_graph_palabras.gexf')

# Crear directorio si no existe
os.makedirs(results_folder, exist_ok=True)

# Export the NetworkX graph G to the GEXF file format
nx.write_gexf(G, gexf_file_path)

# Resumen
n_imgs  = sum(1 for _, d in G.nodes(data=True) if d.get('dimension') == 'imagen')
n_words = sum(1 for _, d in G.nodes(data=True) if d.get('dimension') == 'transcripcion')
print(f'Grafo transcripción: {n_imgs} imágenes · {n_words} palabras · {G.number_of_edges()} aristas')
print(f"NetworkX graph successfully exported to GEXF: {gexf_file_path}")

Grafo transcripción: 97 imágenes · 127 palabras · 156 aristas
NetworkX graph successfully exported to GEXF: drive/MyDrive/TFM-Sara/output/transcripciones/knowledge_graph_palabras.gexf


In [ ]:
import json
import os

# 2. Create an empty dictionary for React graph data
react_graph_data = {
    'nodes': [],
    'links': []
}

# 3. Populate the 'nodes' list
for node_id, attrs in G.nodes(data=True):
    node_data = {
        'id': node_id,
        'label': node_id, # Default label to node_id
        **attrs # Include all other attributes
    }
    react_graph_data['nodes'].append(node_data)

# 4. Populate the 'links' list
for u, v, data in G.edges(data=True):
    react_graph_data['links'].append({'source': u, 'target': v, **data})

# 5. Construct the full path for the output JSON file
json_file_path = os.path.join(results_folder, 'knowledge_graph.json')

# 6. Save the react_graph_data dictionary to the specified JSON file
with open(json_file_path, 'w', encoding='utf-8') as f:
    json.dump(react_graph_data, f, indent=4, ensure_ascii=False)

# 7. Print a confirmation message
print(f"NetworkX graph successfully exported to React-friendly JSON: {json_file_path}")

NetworkX graph successfully exported to React-friendly JSON: drive/MyDrive/TFM-Sara/output/transcripciones/knowledge_graph.json
